In [7]:
# Phase 1: Data Pipeline
from pathlib import Path

import pandas as pd

# 1. Load the project dataset from either the workspace root or notebook folder.
workspace_root = Path.cwd()
data_path = workspace_root / "Kaggle_Ecommerce Data.csv"
if not data_path.exists():
    data_path = workspace_root.parent / "Kaggle_Ecommerce Data.csv"

if not data_path.exists():
    raise FileNotFoundError("Kaggle_Ecommerce Data.csv was not found in the workspace or its parent folder.")

df = pd.read_csv(data_path, encoding="latin1")
print("Data loaded with shape:", df.shape)

# 2. Validate the schema already provided by this dataset.
required_columns = {
    "order_id", "customer_id", "product_id", "price", "quantity",
    "order_date", "delivered_date", "region", "returned", "return_reason",
    "request_date", "total_amount"
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

# 3. Clean identifiers and dates without replacing the source return information.
df = df.dropna(subset=["order_id", "customer_id", "product_id"]).copy()
for column in ["order_date", "delivered_date", "request_date"]:
    df[column] = pd.to_datetime(df[column], errors="coerce", dayfirst=True)

df["region"] = df["region"].fillna("Unknown")
df["return_reason"] = df["return_reason"].fillna("Not returned")
df["is_return"] = df["returned"].astype(str).str.strip().str.lower().eq("yes").astype(int)

# 4. Final quality checks and output.
if df["order_id"].duplicated().any():
    print("Note: repeated order IDs are retained because an order may contain multiple products.")

print("Return records:", int(df["is_return"].sum()))
print(df.head(10))

output_path = data_path.parent / "cleaned_ecommerce_returns.csv"
df.to_csv(output_path, index=False)
print("Cleaned dataset saved as:", output_path)

Data loaded with shape: (34500, 19)
Return records: 1903
  order_id customer_id product_id     category   price  discount  quantity  \
0  O100000      C17270    P234890         Home  164.08      0.15         1   
1  O100001      C17603    P228204      Grocery   24.73      0.00         1   
2  O100002      C10860    P213892  Electronics  175.58      0.05         1   
3  O100003      C15390    P208689  Electronics   63.67      0.00         1   
4  O100004      C15226    P228063         Home   16.33      0.15         1   
5  O100005      C15191    P214062       Beauty   53.91      0.10         2   
6  O100006      C13772    P201363  Electronics  266.50      0.00         1   
7  O100007      C13092    P216691       Beauty    9.98      0.00         1   
8  O100008      C15734    P202751      Fashion    6.61      0.05         1   
9  O100009      C16265    P207782      Grocery   10.91      0.00         1   

  payment_method order_date delivered_date region returned request_date  \
0    Cred